In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pathlib import Path
from harbor.analysis.cross_docking import DockingDataModel
from plotting_params import *
from importlib import reload
reload(p)

## input files

In [ ]:
posit_raw = DockingDataModel.deserialize("/Users/alexpayne/Scientific_Projects/mers-drug-discovery/sars2-retrospective-analysis/ALL_combined_results.parquet")

In [ ]:
posit_results = Path("/Users/alexpayne/Scientific_Projects/mers-drug-discovery/sars2-retrospective-analysis/analyzed_results/all_evals_posit_combined_results.csv")

In [ ]:
pdf = pd.read_csv(posit_results)
pdf["Error_Lower"] = pdf["Fraction"] - pdf["CI_Lower"]
pdf["Error_Lower"] = pdf["Error_Lower"].apply(lambda x: 0 if x < 0 else x)
pdf["Error_Upper"] = pdf["CI_Upper"] - pdf["Fraction"]
pdf["Error_Upper"] = pdf["Error_Upper"].apply(lambda x: 0 if x < 0 else x)

In [ ]:
fdf = pd.read_csv("/Users/alexpayne/Scientific_Projects/mers-drug-discovery/sars2-retrospective-analysis/analyzed_results/all_evals_fred_combined_results.csv")

## output files

In [ ]:
figpath = Path("../figures")
figpath.mkdir(exist_ok=True)

# Reference Split Comparison

In [ ]:
df = pdf[(~pdf["Reference_Split"].isna())&(pdf["PairwiseSplit"].isna())]

In [ ]:
df = df.groupby(["Reference_Split", "Score", "N_Reference_Structures"]).head(1)

In [ ]:
ALPHA = 0.3
p.figure_decorator(func=plot_filled_in_error_bars, label_map=label_map, fig_path=figpath / "dataset_split", raw_df=df)
ALPHA = 0.2

In [ ]:
def plot_filled_in_error_bars_facet_col(
    raw_df,
    x_var=X_VAR,
    y_var=Y_VAR,
    color_var=COLOR_VAR,
    style_var=STYLE_VAR,
    facet_var=STYLE_VAR,  # New parameter, defaults to None
    ci_lower=CI_LOWER,
    ci_upper=CI_UPPER,
):
    """Plot filled-in error bars with facets by specified variable"""
    # Use style_var as facet_var if none provided
    facet_var = facet_var or style_var
    style_var = style_var or color_var

    # Sort the dataframe
    raw_df = raw_df.sort_values(by=[x_var, style_var, color_var])

    # Create subplot for each facet value
    facets = raw_df[facet_var].unique()
    fig, axes = plt.subplots(1, len(facets), figsize=(LARGE_FIG_SIZE[0]*len(facets), LARGE_FIG_SIZE[1]))

    # Create color mapping
    unique_colors = sns.color_palette(n_colors=len(raw_df[color_var].unique()))
    color_map = dict(zip(sorted(raw_df[color_var].unique()), unique_colors))

    for ax, facet in zip(axes, facets):
        facet_data = raw_df[raw_df[facet_var] == facet]

        # Create fill between for each group using matched colors
        for name, group in facet_data.groupby([color_var, style_var]):
            color_name = name[0]  # First element is Score
            ax.fill_between(
                group[x_var],
                group[ci_lower],
                group[ci_upper],
                color=color_map[color_name],
                alpha=ALPHA,
            )

        # Create the line plot
        sns.lineplot(
            data=facet_data,
            x=x_var,
            y=y_var,
            hue=color_var,
            style=style_var,  # Keep style_var for line styles
            ax=ax,
            palette=color_map,
            hue_order=list(reversed(sorted(raw_df[color_var].unique()))),
            style_order=list(reversed(sorted(raw_df[style_var].unique())))
        )
        
        # Customize each subplot
        ax.set_xscale("log")
        ax.xaxis.set_major_formatter(ScalarFormatter())

        custom_ticks = [1, 5, 10, 20, 50, 100, 200, raw_df[x_var].max()]
        ax.set_xticks(custom_ticks)
        ax.set_xticklabels(custom_ticks, fontsize=FONT_SIZES["ticks"])
        ax.tick_params(axis='y', labelsize=FONT_SIZES["ticks"])

        ax.set_xlabel(X_LABEL, fontsize=FONT_SIZES["xlabel"], fontweight="bold")
        if ax == axes[0]:  # Only set ylabel for first subplot
            ax.set_ylabel(Y_LABEL, fontsize=FONT_SIZES["ylabel"], fontweight="bold")
        else:
            ax.set_ylabel("")

        ax.set_title(f"{facet}", fontsize=FONT_SIZES["xlabel"], fontweight="bold")

        # Customize legend
        legend = ax.legend()
        plt.setp(legend.get_title(), fontsize=FONT_SIZES["legend_title"], fontweight="bold")
        plt.setp(legend.get_texts(), fontsize=FONT_SIZES["legend_text"])
    return plt

In [ ]:
figure_decorator(plot_filled_in_error_bars_facet_col,color_var=STYLE_VAR, label_map=label_map,facet_var=COLOR_VAR, raw_df=df, fig_path=figpath / "dataset_split_wide")

### is RMSD datesplit actually better than RandomSplit?

In [ ]:
from harbor.analysis.cross_docking import EvaluatorFactory, Results, Evaluator

In [ ]:
evf = EvaluatorFactory(name="test")
evf.reference_split_settings.use = True
evf.reference_split_settings.date_split_settings.use = True
evf.reference_split_settings.date_split_settings.reference_structure_date_column = (
    "Reference_Structure_Date"
)
evf.reference_split_settings.random_split_settings.use = True
evf.scorer_settings.rmsd_scorer_settings.use = True
evf.scorer_settings.posit_scorer_settings.use = False

evf.reference_split_settings.n_reference_structures = [100]
evf.n_bootstraps = 1
evs: [Evaluator] = evf.create_evaluators(posit_raw)

In [ ]:
random = evs[0]
date = evs[1]
raw = random.run_pose_selector(posit_raw)

In [ ]:
testdf = raw.dataframe
testdf = testdf.sort_values("RMSD", ascending=True).groupby(["Query_Ligand"]).head(1)

In [ ]:
test_scoring = DockingDataModel(dataframe=testdf, **raw.model_dump())

In [ ]:
scored = random.calculate_results([test_scoring])

In [ ]:
evf.reference_split_settings.n_reference_structures = [100]
evf.n_bootstraps = 1
evs: [Evaluator] = evf.create_evaluators(posit_raw)
random = evs[0]
date = evs[1]
rdf = random.run_dataset_split(raw)[0].dataframe
ddf = date.run_dataset_split(raw)[0].dataframe

simdf = pd.concat([pd.DataFrame({"Split": "RandomSplit", "Max MCS Tanimoto":rdf[rdf["Type"] == "MCS"].groupby("Query_Ligand")["Tanimoto"].max(), "Min MCS Tanimoto":rdf[rdf["Type"] == "MCS"].groupby("Query_Ligand")["Tanimoto"].min()}),
                   pd.DataFrame({"Split": "DateSplit", "Max MCS Tanimoto":ddf[ddf["Type"] == "MCS"].groupby("Query_Ligand")["Tanimoto"].max(), "Min MCS Tanimoto":ddf[ddf["Type"] == "MCS"].groupby("Query_Ligand")["Tanimoto"].min()}),
                  ])

In [ ]:
from scipy import stats
import numpy as np
def plot_filled_ecdf_minmax(data, x_min_column, x_max_column, hue_column, complementary=True, alpha=0.2):
    # Get unique values for the hue column
    hue_values = data[hue_column].unique()

    # Create figure
    plt.figure(figsize=SMALL_FIG_SIZE)

    ecdfs = {}
    # Calculate ECDFs for both min and max for each split
    for hue in hue_values:
        subset_min = data[data[hue_column] == hue][x_min_column]
        subset_max = data[data[hue_column] == hue][x_max_column]
        
        # Calculate ECDFs
        ecdf_min = stats.ecdf(subset_min)
        ecdf_max = stats.ecdf(subset_max)
        
        x_min = np.sort(subset_min)
        x_max = np.sort(subset_max)
        
        y_min = ecdf_min.cdf.evaluate(x_min)
        y_max = ecdf_max.cdf.evaluate(x_max)

        if complementary:
            y_min = 1 - y_min
            y_max = 1 - y_max

        ecdfs[hue] = {
            'min': (x_min, y_min),
            'max': (x_max, y_max)
        }

    # Plot for each split type
    for hue in hue_values:
        x_min, y_min = ecdfs[hue]['min']
        x_max, y_max = ecdfs[hue]['max']
        
        # Create common x values for fill_between
        x_all = np.unique(np.concatenate([x_min, x_max]))
        y_min_interp = np.interp(x_all, x_min, y_min)
        y_max_interp = np.interp(x_all, x_max, y_max)

        # Fill between min and max curves
        plt.fill_between(x_all, y_min_interp, y_max_interp, alpha=alpha, label=hue)

        # Plot the boundary lines
        # plt.plot(x_min, y_min, color=plt.gca().lines[-1].get_color())
        # plt.plot(x_max, y_max, color=plt.gca().lines[-1].get_color())

    plt.xlabel("MCS Tanimoto")
    plt.ylabel("ECDF")
    plt.legend()
    return plt

In [ ]:
plt = plot_filled_ecdf_minmax(simdf,
                       x_min_column="Min MCS Tanimoto",
                       x_max_column="Max MCS Tanimoto",
                       hue_column="Split",
                       complementary=True,
                              alpha=0.5)
plt = update_labels(plt, label_map, x_label="MCS Tanimoto Range to the References", y_label="Fraction of Molecules", legend_title="Reference Split")
save_figure(plt, figpath / "mcs_tanimoto_range")

In [ ]:
rdf = random.run_dataset_split(raw)[0].dataframe
testdf = rdf.sort_values("RMSD", ascending=True).groupby(["Query_Ligand"]).head(1)
test_scoring = DockingDataModel(dataframe=testdf, **raw.model_dump())
scored = random.calculate_results([test_scoring])
print(scored)

In [ ]:
ddf = date.run_dataset_split(raw)[0].dataframe
testdf = ddf.sort_values("RMSD", ascending=True).groupby(["Query_Ligand"]).head(1)
test_scoring = DockingDataModel(dataframe=testdf, **raw.model_dump())
scored = random.calculate_results([test_scoring])
print(scored)

In [ ]:
results = Results.df_from_results(Results.calculate_results(posit_raw, evs))

In [ ]:
results

In [ ]:
prdf = posit_raw.dataframe

In [ ]:
refs = prdf.Reference_Structure.unique()[:100]